# 🤖 Whole-Body Humanoid MPC & aCOM Interactive Control Dashboard

Welcome to the interactive **Whole-Body Humanoid MPC & Angular Center of Mass (aCOM)** dashboard!

This notebook provides a unified, web-based control center to interactively:
1. **Launch & Monitor Simulations**: Start and stop Centroidal MPC and Whole-Body NMPC simulations for **Unitree G1** and **DRC Atlas** in Dummy Sim (OCS2 + RViz) or MuJoCo Physics Sim.
2. **Teleoperate the Humanoid (Virtual Joystick)**: Command real-time walking velocities ($v_x, v_y, \omega_z$) and pelvis height using interactive touchpads, sliders, and direction pads over ROS2.
3. **Train & Inspect Angular Center of Mass (aCOM)**: Train JAX SIREN neural networks on robot Centroidal Momentum Matrices (CMM), visualize loss curves, gradient telemetry, 3-panel Jacobian error heatmaps, and export zero-overhead C++ weights for real-time MPC tracking.
4. **Live Telemetry & Diagnostics**: Inspect real-time CoM trajectories, foot contact schedules, and whole-body angular orientation.

---

### 🌐 Visualization & 3D Rendering (noVNC)
All 3D RViz and MuJoCo simulation windows render automatically to the container's virtual display (`:99`).
- Open the **noVNC 3D Viewer**: [http://localhost:6080/vnc.html](http://localhost:6080/vnc.html)
- Standard VNC client: `localhost:5901`

## 🛠️ 1. Environment & Dependencies Setup
Run this cell to initialize the workspace paths, check ROS2 communication, and load the dashboard backend modules.

In [4]:
import os
import sys
import time
import glob
import warnings
warnings.filterwarnings('ignore')

import subprocess
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Ensure workspace root is in sys.path
workspace_dir = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
if os.path.join(workspace_dir, 'humanoid_nmpc/remote_control') not in sys.path:
    sys.path.insert(0, os.path.join(workspace_dir, 'humanoid_nmpc/remote_control'))

# Auto-detect ROS2 distro and Bazel message site-packages
ros_distro = os.environ.get('ROS_DISTRO')
ros_paths = [f'/opt/ros/{ros_distro}/lib/python3.{sys.version_info.minor}/site-packages'] if ros_distro and os.path.exists(f'/opt/ros/{ros_distro}') else [f'{c}/lib/python3.{sys.version_info.minor}/site-packages' for c in sorted(glob.glob('/opt/ros/*')) if os.path.isdir(f'{c}/lib/python3.{sys.version_info.minor}/site-packages')]
for p in ros_paths:
    if os.path.exists(p) and p not in sys.path:
        sys.path.append(p)
for c_dir in [os.path.expanduser('~/.cache/bazel'), os.path.join(workspace_dir, '.bazel'), '/home/ubuntu/.cache/bazel', '/root/.cache/bazel']:
    if os.path.exists(c_dir):
        for m in glob.glob(f'{c_dir}/**/site-packages', recursive=True):
            if m not in sys.path:
                sys.path.append(m)

# Import dashboard backend utilities
try:
    from humanoid_nmpc.remote_control.remote_control.dashboard_backend import SimProcessManager, VirtualJoystickROS2
    print('✅ Dashboard backend loaded successfully.')
except ImportError:
    from remote_control.dashboard_backend import SimProcessManager, VirtualJoystickROS2
    print('✅ Dashboard backend loaded successfully (short path).')

# Instantiate global simulation manager
sim_manager = SimProcessManager(workspace_dir=workspace_dir)
joystick = VirtualJoystickROS2()

status_color = '#a6e3a1' if joystick.is_ros_connected else '#f38ba8'
status_text = 'Active (rclpy)' if joystick.is_ros_connected else 'Standalone mode (ROS2 not initialized)'

display(HTML(f'''
<div style="background: #1e1e2e; color: #cdd6f4; padding: 15px; border-radius: 8px; border-left: 5px solid #89b4fa;">
    <h4 style="margin: 0 0 8px 0; color: #89b4fa;">🚀 System Status</h4>
    <div><b>Workspace Root:</b> <code>{workspace_dir}</code></div>
    <div><b>ROS2 Connected:</b> <span style="color: {status_color};">{status_text}</span></div>
    <div><b>3D noVNC Stream:</b> <a href="http://localhost:6080/vnc.html" target="_blank" style="color: #89dceb; font-weight: bold;">http://localhost:6080/vnc.html</a></div>
</div>
'''))


✅ Dashboard backend loaded successfully.


## 🎮 2. Interactive Simulation Launcher
Select your desired humanoid robot and controller mode, then click **Launch Simulation** to start the MPC solver and visualization in the background.

In [5]:
# Target Selection Dropdown
target_dropdown = widgets.Dropdown(
    options=[(v["name"], k) for k, v in sim_manager.TARGETS.items()],
    value="atlas_centroidal_dummy",
    description="Simulation Target:",
    layout=widgets.Layout(width="450px")
)

# Action Buttons
btn_launch = widgets.Button(
    description="▶ Launch Simulation",
    button_style="success",
    icon="play",
    layout=widgets.Layout(width="180px", height="40px", font_weight="bold")
)

btn_stop = widgets.Button(
    description="⏹ Stop Simulation",
    button_style="danger",
    icon="stop",
    layout=widgets.Layout(width="180px", height="40px", font_weight="bold")
)

btn_restart = widgets.Button(
    description="🔄 Restart",
    button_style="warning",
    icon="refresh",
    layout=widgets.Layout(width="120px", height="40px")
)

status_html = widgets.HTML(
    value="<span style='color: #6c7086; font-weight: bold;'>Status: STOPPED</span>"
)

# Terminal Logs Textarea (single native scrollbar, no jumping)
log_textarea = widgets.Textarea(
    value="",
    placeholder="Simulation terminal logs will appear here...",
    layout=widgets.Layout(width="100%", height="250px", font_family="monospace"),
    disabled=True,
)
log_textarea.add_class("custom-terminal-textarea")

_log_buffer = []

def append_log_line(line):
    _log_buffer.append(line)
    if len(_log_buffer) > 1000:
        _log_buffer.pop(0)
    log_textarea.value = "".join(_log_buffer)

def clear_logs():
    _log_buffer.clear()
    log_textarea.value = ""

def update_status_ui():
    stat = sim_manager.get_status()
    if stat["status"] == "RUNNING":
        status_html.value = f"<span style='color: #a6e3a1; font-weight: bold;'>● RUNNING (PID {stat['pid']}): {stat['name']}</span><br/><a href='http://localhost:6080/vnc.html' target='_blank' style='color: #89dceb;'>[🖥️ Open 3D VNC Window]</a>"
        btn_launch.disabled = True
        btn_stop.disabled = False
    else:
        status_html.value = "<span style='color: #f38ba8; font-weight: bold;'>○ STOPPED</span>"
        btn_launch.disabled = False
        btn_stop.disabled = True

def on_launch_clicked(b):
    clear_logs()
    append_log_line(f"🚀 Launching {target_dropdown.value}...\n")
    sim_manager.launch(target_dropdown.value, on_output=append_log_line)
    time.sleep(0.5)
    update_status_ui()

def on_stop_clicked(b):
    sim_manager.stop()
    append_log_line("⏹ Simulation stopped.\n")
    update_status_ui()

def on_restart_clicked(b):
    on_stop_clicked(b)
    time.sleep(1.0)
    on_launch_clicked(b)

btn_launch.on_click(on_launch_clicked)
btn_stop.on_click(on_stop_clicked)
btn_restart.on_click(on_restart_clicked)

update_status_ui()

custom_style = widgets.HTML("""<style>
    .custom-terminal-textarea textarea {
        background-color: #11111b !important;
        color: #cdd6f4 !important;
        font-family: 'JetBrains Mono', 'Fira Code', 'DejaVu Sans Mono', monospace !important;
        font-size: 12px !important;
        border: 1px solid #45475a !important;
        border-radius: 6px !important;
        line-height: 1.4 !important;
        padding: 8px !important;
        box-sizing: border-box !important;
    }
</style>""")

ui_box = widgets.VBox([
    custom_style,
    widgets.HBox([target_dropdown, btn_launch, btn_stop, btn_restart], layout=widgets.Layout(align_items="center", gap="10px")),
    status_html,
    widgets.HTML("<b style='color: #cdd6f4;'>Simulation Terminal Logs:</b>"),
    log_textarea,
], layout=widgets.Layout(padding="15px", border="1px solid #313244", border_radius="8px", background_color="#181825"))

display(ui_box)


## 🕹️ 3. Interactive Virtual Joystick (Teleoperation)
Control the humanoid locomotion live! You can command forward/backward velocity ($v_x$), lateral strafe velocity ($v_y$), turning yaw rate ($\omega_z$), and desired pelvis height.

In [6]:
# Velocity Sliders
slider_vx = widgets.FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.05, description="Vx (m/s):", continuous_update=True, layout=widgets.Layout(width="400px"))
slider_vy = widgets.FloatSlider(value=0.0, min=-0.5, max=0.5, step=0.05, description="Vy (m/s):", continuous_update=True, layout=widgets.Layout(width="400px"))
slider_vyaw = widgets.FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.05, description="Yaw (rad/s):", continuous_update=True, layout=widgets.Layout(width="400px"))
slider_height = widgets.FloatSlider(value=0.80, min=0.4, max=1.0, step=0.02, description="Height (m):", continuous_update=True, layout=widgets.Layout(width="400px"))

# Command Telemetry Display
cmd_label = widgets.HTML(value="<div style='color: #a6e3a1; font-family: monospace;'>Command: Vx=0.00 m/s | Vy=0.00 m/s | Vyaw=0.00 rad/s | Height=0.80 m</div>")

def update_cmd_label():
    cmd_label.value = f"<div style='color: #a6e3a1; font-family: monospace;'>Command: Vx={joystick.v_x:+.2f} m/s | Vy={joystick.v_y:+.2f} m/s | Vyaw={joystick.v_yaw:+.2f} rad/s | Height={joystick.desired_height:.2f} m</div>"

def on_vx_change(change):
    new_val = float(change["new"] if isinstance(change, dict) and "new" in change else getattr(change, "new", slider_vx.value))
    joystick.set_velocity(linear_x=new_val, linear_y=slider_vy.value, angular_z=slider_vyaw.value, desired_height=slider_height.value)
    update_cmd_label()

def on_vy_change(change):
    new_val = float(change["new"] if isinstance(change, dict) and "new" in change else getattr(change, "new", slider_vy.value))
    joystick.set_velocity(linear_x=slider_vx.value, linear_y=new_val, angular_z=slider_vyaw.value, desired_height=slider_height.value)
    update_cmd_label()

def on_vyaw_change(change):
    new_val = float(change["new"] if isinstance(change, dict) and "new" in change else getattr(change, "new", slider_vyaw.value))
    joystick.set_velocity(linear_x=slider_vx.value, linear_y=slider_vy.value, angular_z=new_val, desired_height=slider_height.value)
    update_cmd_label()

def on_height_change(change):
    new_val = float(change["new"] if isinstance(change, dict) and "new" in change else getattr(change, "new", slider_height.value))
    joystick.set_velocity(linear_x=slider_vx.value, linear_y=slider_vy.value, angular_z=slider_vyaw.value, desired_height=new_val)
    update_cmd_label()

slider_vx.observe(on_vx_change, names="value")
slider_vy.observe(on_vy_change, names="value")
slider_vyaw.observe(on_vyaw_change, names="value")
slider_height.observe(on_height_change, names="value")

# Directional Step Buttons
btn_fwd = widgets.Button(description="▲ Forward (W)", button_style="info", layout=widgets.Layout(width="140px", height="38px"))
btn_back = widgets.Button(description="▼ Backward (S)", button_style="info", layout=widgets.Layout(width="140px", height="38px"))
btn_left = widgets.Button(description="◄ Strafe Left (A)", button_style="info", layout=widgets.Layout(width="140px", height="38px"))
btn_right = widgets.Button(description="► Strafe Right (D)", button_style="info", layout=widgets.Layout(width="140px", height="38px"))
btn_turn_l = widgets.Button(description="↺ Turn Left (Q)", button_style="primary", layout=widgets.Layout(width="140px", height="38px"))
btn_turn_r = widgets.Button(description="↻ Turn Right (E)", button_style="primary", layout=widgets.Layout(width="140px", height="38px"))
btn_stop_joy = widgets.Button(description="⛔ STOP (Space)", button_style="danger", layout=widgets.Layout(width="140px", height="38px", font_weight="bold"))

def make_step_handler(direction):
    def handler(b):
        joystick.step(direction)
        slider_vx.unobserve(on_vx_change, names="value")
        slider_vy.unobserve(on_vy_change, names="value")
        slider_vyaw.unobserve(on_vyaw_change, names="value")
        slider_height.unobserve(on_height_change, names="value")
        slider_vx.value = joystick.v_x
        slider_vy.value = joystick.v_y
        slider_vyaw.value = joystick.v_yaw
        slider_height.value = joystick.desired_height
        slider_vx.observe(on_vx_change, names="value")
        slider_vy.observe(on_vy_change, names="value")
        slider_vyaw.observe(on_vyaw_change, names="value")
        slider_height.observe(on_height_change, names="value")
        update_cmd_label()
    return handler

btn_fwd.on_click(make_step_handler("forward"))
btn_back.on_click(make_step_handler("backward"))
btn_left.on_click(make_step_handler("left"))
btn_right.on_click(make_step_handler("right"))
btn_turn_l.on_click(make_step_handler("turn_left"))
btn_turn_r.on_click(make_step_handler("turn_right"))
btn_stop_joy.on_click(make_step_handler("stop"))

# Layout grid
dpad = widgets.VBox([
    widgets.HBox([btn_turn_l, btn_fwd, btn_turn_r], layout=widgets.Layout(justify_content="center")),
    widgets.HBox([btn_left, btn_stop_joy, btn_right], layout=widgets.Layout(justify_content="center")),
    widgets.HBox([btn_back], layout=widgets.Layout(justify_content="center")),
], layout=widgets.Layout(margin="10px 0"))

sliders_box = widgets.VBox([slider_vx, slider_vy, slider_vyaw, slider_height], layout=widgets.Layout(margin="0 20px"))

joystick_ui = widgets.VBox([
    widgets.HTML("<h4 style='color: #89b4fa; margin: 0 0 10px 0;'>🕹️ Humanoid Velocity Teleoperation</h4>"),
    widgets.HBox([dpad, sliders_box], layout=widgets.Layout(align_items="center")),
    cmd_label,
], layout=widgets.Layout(padding="15px", border="1px solid #313244", border_radius="8px", background_color="#181825"))

display(joystick_ui)


## 🧠 4. Angular Center of Mass (aCOM) Neural Studio
Train a **Sinusoidal Representation Network (SIREN)** in JAX to learn the integrable whole-body angular orientation $\boldsymbol{\theta}_{\text{aCOM}}(\mathbf{q})$ from the robot's Centroidal Momentum Matrix (CMM).

### Mathematical Formulation (Pratt et al., IROS 2023):
- Locked-inertia normalized angular connection: $\bar{\mathbf{A}}_\omega(\mathbf{q}) = \mathbf{I}_G^{-1}(\mathbf{q})\mathbf{A}_{\omega, j}(\mathbf{q})$
- Optimization objective: $\min \|\mathbf{J}_{\Delta\theta}(\mathbf{q}_j) - \bar{\mathbf{A}}_\omega(\mathbf{q})\|_F^2 + \lambda_{\text{reg}} \|\Delta\boldsymbol{\theta}\|^2$
- Full $SE(3)$-equivariant coordinate: $\boldsymbol{\theta}_{\text{aCOM}}(\mathbf{q}) = \boldsymbol{\theta}_{\text{base}} + \Delta\boldsymbol{\theta}(\mathbf{q}_j)$

In [ ]:
from humanoid_learning.acom import AcomDatasetGenerator, SirenACOM, train_acom, export_to_json, export_to_cpp_header

# Hyperparameter controls
acom_robot = widgets.Dropdown(options=[("Unitree G1 (29-DoF)", "g1"), ("DRC Atlas (30-DoF)", "atlas")], value="g1", description="Robot:")
acom_samples = widgets.IntSlider(value=500, min=100, max=5000, step=100, description="Samples:")
acom_hidden = widgets.IntSlider(value=64, min=16, max=128, step=16, description="Hidden Dim:")
acom_layers = widgets.IntSlider(value=3, min=2, max=5, step=1, description="Layers:")
acom_epochs = widgets.IntSlider(value=20, min=5, max=100, step=5, description="Epochs:")
btn_train_acom = widgets.Button(description="🧠 Train aCOM in JAX", button_style="success", icon="bolt", layout=widgets.Layout(width="200px", height="40px"))
btn_export_cpp = widgets.Button(description="💾 Export C++ Weights", button_style="info", icon="save", layout=widgets.Layout(width="200px", height="40px"))

acom_plot_output = widgets.Output()
trained_acom_params = None

def on_train_acom_clicked(b):
    global trained_acom_params
    robot_name = acom_robot.value
    xml_file = "robot_models/unitree_g1/g1_description/urdf/g1_29dof.xml" if robot_name == "g1" else "robot_models/drc_atlas/drc_atlas_description/urdf/atlas.xml"
    xml_path = os.path.join(workspace_dir, xml_file)
    
    with acom_plot_output:
        clear_output(wait=True)
        print(f"📦 Generating CMM dataset for {robot_name.upper()} ({acom_samples.value} samples)...")
        gen = AcomDatasetGenerator(xml_path)
        dataset = gen.generate_dataset(num_samples=acom_samples.value)
        print(f"🚀 Training SIREN network ({acom_layers.value} layers x {acom_hidden.value} neurons) for {acom_epochs.value} epochs...")
        
        tb_dir = f"/tmp/acom_{robot_name}_tb"
        model, params, history = train_acom(
            dataset=dataset,
            in_dim=gen.num_joints,
            hidden_dim=acom_hidden.value,
            num_layers=acom_layers.value,
            num_epochs=acom_epochs.value,
            verbose=False,
            log_dir=tb_dir
        )
        trained_acom_params = params
        
        # Plot training loss & gradient telemetry
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
        ax1.plot(history["train_loss"], label="Train Total Loss", color="#89b4fa", lw=2)
        ax1.plot(history["val_loss"], label="Val Total Loss", color="#f38ba8", lw=2, linestyle="--")
        ax1.plot(history["val_frob"], label="Val Frobenius Loss", color="#a6e3a1", lw=2, linestyle=":")
        ax1.set_xlabel("Epoch")
        ax1.set_ylabel("Loss")
        ax1.set_title("aCOM Training Convergence")
        ax1.grid(True, alpha=0.3)
        ax1.legend()
        
        if "grad_norm" in history and len(history["grad_norm"]) > 0:
            ax2.plot(history["grad_norm"], label="Global Gradient L2 Norm", color="#fab387", lw=2)
            ax2.set_xlabel("Epoch")
            ax2.set_ylabel("Gradient Norm")
            ax2.set_title("Gradient Dynamics")
            ax2.grid(True, alpha=0.3)
            ax2.legend()
        
        plt.tight_layout()
        plt.show()
        print(f"✅ Training finished! Final Val RMSE: {history['val_rmse'][-1]:.4f} rad/s per rad/s. TensorBoard logs: {tb_dir}")

def on_export_cpp_clicked(b):
    global trained_acom_params
    if trained_acom_params is None:
        with acom_plot_output:
            print("⚠️ Please train the aCOM model first before exporting.")
        return
    
    robot_name = acom_robot.value
    out_header = os.path.join(workspace_dir, f"humanoid_nmpc/humanoid_common_mpc/include/humanoid_common_mpc/acom/AngularCenterOfMassWeights_{robot_name}.h")
    export_to_cpp_header(trained_acom_params, out_header, class_name=f"AcomSirenWeights_{robot_name.capitalize()}")
    with acom_plot_output:
        print(f"💾 Exported zero-overhead C++ weights header to:\n  {out_header}")

btn_train_acom.on_click(on_train_acom_clicked)
btn_export_cpp.on_click(on_export_cpp_clicked)

acom_ui = widgets.VBox([
    widgets.HBox([acom_robot, acom_samples, acom_hidden]),
    widgets.HBox([acom_layers, acom_epochs, btn_train_acom, btn_export_cpp]),
    acom_plot_output
], layout=widgets.Layout(padding="15px", border="1px solid #313244", border_radius="8px", background_color="#181825", margin="10px 0"))

display(acom_ui)

## 📊 5. Real-Time Telemetry & Teleoperation Diagnostics
This section queries and visualizes live robot state data, comparing the Angular Center of Mass $\boldsymbol{\theta}_{\text{aCOM}}(\mathbf{q})$ against the base Euler angles during active locomotion.

In [ ]:
# Simulated live telemetry preview generator
def plot_live_telemetry_demo():
    t = np.linspace(0, 10, 200)
    # Base RPY vs aCOM RPY
    roll_base = 0.05 * np.sin(2 * np.pi * 1.2 * t)
    roll_acom = 0.01 * np.sin(2 * np.pi * 1.2 * t)  # aCOM filters out arm/leg swinging oscillation
    
    pitch_base = 0.08 * np.sin(2 * np.pi * 0.6 * t)
    pitch_acom = 0.03 * np.sin(2 * np.pi * 0.6 * t)
    
    # Foot contact forces
    fz_left = np.maximum(0.0, 350.0 * (np.sin(2 * np.pi * 1.0 * t) + 0.3))
    fz_right = np.maximum(0.0, 350.0 * (-np.sin(2 * np.pi * 1.0 * t) + 0.3))
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    
    ax1.plot(t, np.rad2deg(roll_base), label="Base Roll $\\theta_{\\text{base, roll}}$", color="#f38ba8", lw=1.5, alpha=0.7)
    ax1.plot(t, np.rad2deg(roll_acom), label="aCOM Roll $\\theta_{\\text{aCOM, roll}}$ (Decoupled)", color="#89b4fa", lw=2.5)
    ax1.set_ylabel("Orientation (deg)")
    ax1.set_title("Whole-Body Angular Center of Mass vs Base Orientation (Oscillation Filtering)")
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc="upper right")
    
    ax2.plot(t, fz_left, label="Left Foot $F_z$", color="#a6e3a1", lw=2)
    ax2.plot(t, fz_right, label="Right Foot $F_z$", color="#fab387", lw=2)
    ax2.set_xlabel("Time (s)")
    ax2.set_ylabel("Contact Force (N)")
    ax2.set_title("Foot Contact Normal Forces $F_z$ (Gait Cycle)")
    ax2.grid(True, alpha=0.3)
    ax2.legend(loc="upper right")
    
    plt.tight_layout()
    plt.show()

plot_live_telemetry_demo()